## Test Our Models!

### 1. Import Test dataset

In [ ]:
import torch, torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

torch.manual_seed(42)

In [ ]:
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.507, 0.487, 0.441), (0.267, 0.256, 0.276))
])

full_test_dataset = torchvision.datasets.CIFAR100(
    root='../CIFAR100', 
    train=False, 
    download=True, 
    transform=transform_test
)

In [ ]:
full_test_loader = DataLoader(
    full_test_dataset, 
    batch_size=64, 
    shuffle=False
)

In [ ]:
head_classes = list(range(0, 33))
middle_classes = list(range(33, 67))
tail_classes = list(range(67, 100))

def create_subset_loader(dataset, target_classes, batch_size=64):
    indices = [i for i, (_, label) in enumerate(dataset) if label in target_classes]
    subset = Subset(dataset, indices)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=2)
    return loader, len(indices)

In [ ]:
head_test_loader, head_count = create_subset_loader(full_test_dataset, head_classes)
middle_test_loader, middle_count = create_subset_loader(full_test_dataset, middle_classes)
tail_test_loader, tail_count = create_subset_loader(full_test_dataset, tail_classes)

### Import Model

In [ ]:
from models.model import *
from data.dataset import *
_, _, cls_num_list = get_dataloader(split=False)
base_balanced_model     = CifarResNet18().to(device);                          base_balanced_model.load_state_dict(torch.load("./models_path/Base_balanced.pth"))
base_unbalanced_model   = CifarResNet18().to(device);                          base_unbalanced_model.load_state_dict(torch.load("./models_path/Base_unbalanced.pth"))
ldam_balanced_model     = CifarResNet18(use_norm=True).to(device);             ldam_balanced_model.load_state_dict(torch.load("./models_path/LDAM_balanced.pth"))
ldam_unbalanced_model   = CifarResNet18(use_norm=True).to(device);             ldam_unbalanced_model.load_state_dict(torch.load("./models_path/LDAM_unbalanced.pth"))
multi_stage_model       = CifarResNet18_ThreeStage(cls_num_list).to(device);   multi_stage_model.load_state_dict(torch.load("./models_path/stage3_CSE.pth"))

### Evaluate Model

In [ ]:
from metrics.metric import acc_stage_model

print("=== Evaluate SupCon-Guided Hybrid Model ===")
acc_stage_model(multi_stage_model, full_test_loader, device)

In [ ]:
print("=== Measuring Head/Middle/Tail Accuracy for Multi-stage Model ===")
head_acc_ldam, head_acc_cse, head_acc_conf, head_acc_soft = acc_stage_model(multi_stage_model, head_test_loader, device, head_tail=True)
midd_acc_ldam, midd_acc_cse, midd_acc_conf, midd_acc_soft = acc_stage_model(multi_stage_model, middle_test_loader, device, head_tail=True)
tail_acc_ldam, tail_acc_cse, tail_acc_conf, tail_acc_soft = acc_stage_model(multi_stage_model, tail_test_loader, device, head_tail=True)

print(f"[LDAM strategy]       head acc: {head_acc_ldam:.4f}% | middle acc: {midd_acc_ldam:.4f}% | tail acc : {tail_acc_ldam:.4f}%")
print(f"[cse strategy]        head acc: {head_acc_cse:.4f}% | middle acc: {midd_acc_cse:.4f}% | tail acc : {tail_acc_cse:.4f}%")
print(f"[Confidence strategy] head acc: {head_acc_conf:.4f}% | middle acc: {midd_acc_conf:.4f}% | tail acc : {tail_acc_conf:.4f}%")
print(f"[Softgate strategy]   head acc: {head_acc_soft:.4f}% | middle acc: {midd_acc_soft:.4f}% | tail acc : {tail_acc_soft:.4f}%")

In [ ]:
from metrics.metric import top_1_metric, relative_accuracy

print("=== Evaluate Single-Stage Models ===")
print("Balanced: trained at balanced dataset, Unbalanced: trained at unbalanced dataset")
base_balanced_acc = top_1_metric(base_balanced_model, full_test_loader, device)
base_unbalanced_acc = top_1_metric(base_unbalanced_model, full_test_loader, device)
ldam_balanced_acc = top_1_metric(ldam_balanced_model, full_test_loader, device)
ldam_unbalanced_acc = top_1_metric(ldam_unbalanced_model, full_test_loader, device)

print(f"[Base Model(CSE)] Balanced {base_balanced_acc}% | Unbalanced {base_unbalanced_acc}%")
print(f"[LDAM Model]      Balanced {ldam_balanced_acc}% | Unbalanced {ldam_unbalanced_acc}%")

In [ ]:
def top_1_metric(model, test_loader, device, use_scl, inference_mode = None):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            if use_scl and isinstance(images, (list, tuple)):
                images = images[0].to(device)
            else:
                images = images.to(device)
            labels = labels.to(device)
            if hasattr(model, 'training_stage'):
                outputs = model(images, inference_mode=inference_mode)
            elif hasattr(model, 'use_scl') and model.use_scl:
                _, outputs = model(images)
            else:
                outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            if torch.cuda.is_available():
                correct += (predicted.cpu() == labels.cpu()).sum()
            else:
                correct += (predicted == labels).sum()
    
    accuracy = 100 * correct.item() / total
    return accuracy